# 🗺️ faultmap — Interactive Embedding Space Visualizer

> **faultmap** automatically discovers WHERE your LLM is failing and WHY,
> using embedding-space clustering + statistical hypothesis testing.

This notebook is **fully self-contained** — it generates a synthetic dataset
of 1 536-dimensional prompt embeddings (no API key needed), reduces them to 3D
with UMAP, and renders an interactive Plotly scatter where two dense **failure
slices** float visibly in the space.

Use it as:
- A **visual tour** of what faultmap discovers inside your LLM evaluation
- A **skeleton** you can drop-in replace with your own embeddings and scores
- A **GIF / screenshot source** for READMEs and presentations

---


## Prerequisites

```bash
pip install umap-learn plotly pandas faultmap
```

*Running in Colab? Execute the install cell below first.*


In [1]:
# Uncomment to install in Colab / a fresh environment
!pip install umap-learn plotly pandas faultmap -q
print("Dependencies ready ✓")


Dependencies ready ✓


In [2]:
import numpy as np
import pandas as pd
import umap as umap_mod
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('numpy ', np.__version__)
print('pandas', pd.__version__)


/Users/gabo/Developer/Gabon/faultmap/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


numpy  2.4.4
pandas 3.0.2


---

## Step 1 — Generate Synthetic Data

We create **three semantic clusters** in a 1 536-dimensional embedding space
(same dimensionality as OpenAI `text-embedding-3-small`):

| Cluster | Prompts | Failure Rate | What it represents |
|---------|---------|-------------|-------------------|
| **Legal & Compliance** | 55 | 82 % | Regulatory questions the model can't answer reliably |
| **Multi-Step Debugging** | 50 | 76 % | Complex debugging chains that overwhelm the model |
| **General Questions** | 145 | 12 % | Simple questions the model handles well |

Each failure-slice cluster is a tight Gaussian around a random unit vector
(`spread=0.28`). The background is uniform noise on the unit hypersphere.
UMAP will faithfully preserve this structure in 3D.


In [3]:
np.random.seed(42)
DIM = 1536
THRESHOLD = 0.5

# ── Mock prompt banks ────────────────────────────────────────────────────────
LEGAL = [
    'What are the GDPR requirements for data retention policies?',
    'How do I comply with HIPAA when storing patient data in the cloud?',
    'What penalties apply for CCPA violations in California?',
    'Is our data processing agreement SOC 2 Type II compliant?',
    'How do I handle right-to-erasure requests under GDPR?',
    'What is the difference between a HIPAA covered entity and a business associate?',
    'Do we need a DPA with our EU-based customers?',
    'What are the breach notification timelines under CCPA?',
    'How do I conduct a GDPR Article 35 data protection impact assessment?',
    'When does PCI-DSS scope apply to tokenized payment flows?',
    'What records must we keep under GDPR Article 30 RoPA?',
    'Which EU member states have stricter national derogations than the base GDPR?',
]

DEBUG = [
    'My Kubernetes pod keeps crashing in CrashLoopBackOff. How do I debug this?',
    'CUDA out-of-memory error appears even though my GPU shows 20 GB free.',
    'Why does my PyTorch model output NaN loss after the third training epoch?',
    'My Docker container exits immediately with code 137 — what is wrong?',
    'Postgres is running but psql says connection refused on port 5432.',
    'How do I trace a memory leak in a long-running Python FastAPI service?',
    'My React app builds fine but shows a blank white page in production.',
    'Why does my async Python function sometimes hang indefinitely with no error?',
    'The SSL certificate chain is incomplete — how do I fix it in Nginx config?',
    'My CI pipeline passes locally but always fails in GitHub Actions on pytest.',
    'Celery tasks are not being picked up by the worker even though broker runs.',
    'gRPC service throws DeadlineExceeded under load despite a 30 s timeout.',
]

GENERAL = [
    'How do I reset my password?',
    'What is the pricing for the Pro plan?',
    'Can I export my data to CSV?',
    'How do I invite a teammate to my workspace?',
    'What payment methods do you accept?',
    'Is there a free trial available?',
    'How do I cancel my subscription?',
    'Where can I find the API documentation?',
    'Can I use faultmap with Python 3.11?',
    'What is the difference between HDBSCAN and agglomerative clustering?',
    'How do I increase the max_concurrent_requests setting?',
    'Does faultmap support multilingual prompts and embeddings?',
    'How do I set a custom failure_threshold in SliceAnalyzer?',
    'What embedding models are supported out of the box?',
    'How do I run faultmap without an OpenAI API key?',
]

# ── Helpers ──────────────────────────────────────────────────────────────────
def cluster_vecs(n, seed, spread=0.28):
    rng = np.random.default_rng(seed)
    c = rng.standard_normal(DIM)
    c /= np.linalg.norm(c)
    v = c + rng.standard_normal((n, DIM)) * spread
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    return v.astype(np.float32)

def background_vecs(n):
    rng = np.random.default_rng(7)
    v = rng.standard_normal((n, DIM))
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    return v.astype(np.float32)

def random_scores(n, fail_rate, seed):
    rng = np.random.default_rng(seed)
    return [round(float(rng.uniform(0.05, 0.48) if rng.random() < fail_rate
                        else rng.uniform(0.52, 0.99)), 3) for _ in range(n)]

def tile(bank, n):
    return [bank[i % len(bank)] for i in range(n)]

# ── Assemble ──────────────────────────────────────────────────────────────────
N_L, N_D, N_G = 55, 50, 145

all_vecs     = np.vstack([cluster_vecs(N_L, 1), cluster_vecs(N_D, 2), background_vecs(N_G)])
all_scores   = random_scores(N_L, 0.82, 1) + random_scores(N_D, 0.76, 2) + random_scores(N_G, 0.12, 3)
all_prompts  = tile(LEGAL, N_L) + tile(DEBUG, N_D) + tile(GENERAL, N_G)
all_clusters = ['Legal & Compliance'] * N_L + ['Multi-Step Debugging'] * N_D + ['General Questions'] * N_G
all_status   = ['Fail' if s < THRESHOLD else 'Pass' for s in all_scores]

perm = np.random.default_rng(0).permutation(len(all_scores))
all_vecs     = all_vecs[perm]
all_scores   = [all_scores[i] for i in perm]
all_prompts  = [all_prompts[i] for i in perm]
all_clusters = [all_clusters[i] for i in perm]
all_status   = [all_status[i] for i in perm]

n_fail = all_status.count('Fail')
print(f'Dataset  : {len(all_scores)} prompts | {n_fail} failures ({n_fail/len(all_scores):.1%} baseline)')
print(f'Embedding: {all_vecs.shape}')


Dataset  : 250 prompts | 97 failures (38.8% baseline)
Embedding: (250, 1536)


---

## Step 2 — Dimensionality Reduction with UMAP

UMAP (Uniform Manifold Approximation and Projection) reduces our
**1 536-D** embeddings to **3 dimensions** while preserving local
neighbourhood structure — tight clusters stay tight, distant clusters stay far.

Key parameters:

| Parameter | Value | Effect |
|-----------|-------|--------|
| `n_components` | 3 | Output dimensionality |
| `n_neighbors` | 15 | Neighbourhood size — smaller captures finer structure |
| `min_dist` | 0.1 | Packing — clusters are compact but not collapsed |
| `metric` | `cosine` | Natural for L2-normalised embedding vectors |


In [4]:
print('Running UMAP  1536-D → 3-D ...', end=' ', flush=True)

emb3d = umap_mod.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
    low_memory=False,
).fit_transform(all_vecs)

print('done ✓')
print(f'Reduced shape: {emb3d.shape}')


Running UMAP  1536-D → 3-D ... done ✓
Reduced shape: (250, 3)


---

## Step 3 — 3D Interactive Scatter: Pass / Fail

Each dot is a **prompt** in your evaluation set.

- **Red** = failure (score < 0.5) — the model got this wrong
- **Green** = pass (score ≥ 0.5) — the model got this right

Look for the two **dense red clouds** — those are your failure slices:
coherent input topics where failures cluster statistically above baseline.
The scattered green/gray haze is the low-failure background.

> **Controls:** click-drag to rotate · scroll to zoom · click legend entries to isolate traces


In [6]:
df = pd.DataFrame({
    'x': emb3d[:, 0], 'y': emb3d[:, 1], 'z': emb3d[:, 2],
    'Status':  all_status,
    'Cluster': all_clusters,
    'Score':   all_scores,
    'Prompt':  [p[:90] + '…' if len(p) > 90 else p for p in all_prompts],
})

COLOR_MAP  = {'Pass': '#4ade80', 'Fail': '#f87171'}
SYMBOL_MAP = {
    'Legal & Compliance':  'circle',
    'Multi-Step Debugging': 'square',
    'General Questions':   'diamond',
}

fig = px.scatter_3d(
    df, x='x', y='y', z='z',
    color='Status',
    symbol='Cluster',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    hover_name='Prompt',
    hover_data={
        'Score':   True,
        'Cluster': True,
        'Status':  False,
        'x': False, 'y': False, 'z': False,
    },
    opacity=0.82,
)
fig.update_traces(marker_size=4)

_scene = dict(
    bgcolor='#0d1117',
    xaxis=dict(title='UMAP-1', backgroundcolor='#0d1117',
               gridcolor='#21262d', showbackground=True, color='#8b949e'),
    yaxis=dict(title='UMAP-2', backgroundcolor='#0d1117',
               gridcolor='#21262d', showbackground=True, color='#8b949e'),
    zaxis=dict(title='UMAP-3', backgroundcolor='#0d1117',
               gridcolor='#21262d', showbackground=True, color='#8b949e'),
)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0d1117',
    font=dict(family='monospace', size=12, color='#c9d1d9'),
    title=dict(
        text='faultmap — Prompt Embedding Space (UMAP 3D)',
        font=dict(size=20, color='#58a6ff', family='monospace'), x=0.5,
    ),
    legend=dict(bgcolor='#161b22', bordercolor='#30363d', borderwidth=1, font=dict(size=11)),
    scene=_scene,
    margin=dict(l=0, r=0, t=70, b=0),
    width=1000, height=700,
)
fig.show()


---

## Step 4 — 2D Annotated View

A flat 2D projection makes cluster boundaries easier to screenshot and annotate.
Points are coloured by **cluster membership**; failure-slice centroids are labelled
with their name and observed failure rate.


In [7]:
df['u'] = emb3d[:, 0]
df['v'] = emb3d[:, 1]

CLUSTER_COLOR = {
    'Legal & Compliance':  '#f87171',
    'Multi-Step Debugging': '#fb923c',
    'General Questions':   '#94a3b8',
}

fig2 = px.scatter(
    df, x='u', y='v',
    color='Cluster',
    symbol='Status',
    color_discrete_map=CLUSTER_COLOR,
    symbol_map={'Fail': 'x', 'Pass': 'circle'},
    hover_name='Prompt',
    hover_data={'Score': True, 'Status': True, 'u': False, 'v': False},
    opacity=0.75,
)
fig2.update_traces(marker_size=7)

# Annotate failure-slice centroids
for cluster in ['Legal & Compliance', 'Multi-Step Debugging']:
    mask = df['Cluster'] == cluster
    cx = df.loc[mask, 'u'].mean()
    cy = df.loc[mask, 'v'].mean()
    fail_rate = (df.loc[mask, 'Status'] == 'Fail').mean()
    fig2.add_annotation(
        x=cx, y=cy,
        text=f'<b>{cluster}</b><br>{fail_rate:.0%} failure rate',
        showarrow=True, arrowhead=2, arrowcolor='#f0e68c',
        bgcolor='#161b22', bordercolor='#f0e68c', borderpad=4,
        font=dict(size=11, color='#f0e68c'),
        ax=70, ay=-45,
    )

fig2.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(family='monospace', size=12, color='#c9d1d9'),
    title=dict(text='faultmap — Semantic Clusters (UMAP 2D)',
               font=dict(size=18, color='#58a6ff'), x=0.5),
    legend=dict(bgcolor='#161b22', bordercolor='#30363d', borderwidth=1),
    xaxis=dict(title='UMAP-1', gridcolor='#21262d', color='#8b949e'),
    yaxis=dict(title='UMAP-2', gridcolor='#21262d', color='#8b949e'),
    margin=dict(l=40, r=40, t=70, b=40),
    width=950, height=600,
)
fig2.show()


---

## Step 5 — Failure-Rate Dashboard

The scatter plots show *where* failures cluster in the embedding space.
These two charts quantify *how bad* each slice is:

- **Left** — failure rate per cluster vs the overall baseline
- **Right** — full score distribution, showing the bimodal split in failure slices


In [8]:
CLUSTER_ORDER = ['Legal & Compliance', 'Multi-Step Debugging', 'General Questions']
CLUSTER_COLOR2 = {
    'Legal & Compliance':  '#f87171',
    'Multi-Step Debugging': '#fb923c',
    'General Questions':   '#4ade80',
}

stats = {}
for c in CLUSTER_ORDER:
    mask   = [cl == c for cl in all_clusters]
    c_stat = [s for s, m in zip(all_status, mask) if m]
    c_scr  = [s for s, m in zip(all_scores, mask) if m]
    stats[c] = {
        'fail_rate': c_stat.count('Fail') / len(c_stat),
        'n': len(c_stat),
        'scores': c_scr,
    }

baseline = all_status.count('Fail') / len(all_status)

fig3 = go.Figure()
for c in CLUSTER_ORDER:
    s = stats[c]
    fig3.add_trace(go.Bar(
        x=[c], y=[s['fail_rate']],
        marker_color=CLUSTER_COLOR2[c],
        text=[f'{s["fail_rate"]:.0%}  (n={s["n"]})'],
        textposition='outside',
        textfont=dict(color='#c9d1d9', size=12),
        name=c, showlegend=False,
    ))

fig3.add_hline(
    y=baseline, line_dash='dash', line_color='#f0e68c',
    annotation_text=f'Baseline  {baseline:.1%}',
    annotation_position='top right',
    annotation_font=dict(color='#f0e68c', size=11),
)
fig3.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    title=dict(text='Failure Rate by Semantic Cluster',
               font=dict(size=18, color='#58a6ff'), x=0.5),
    yaxis=dict(title='Failure Rate', tickformat='.0%',
               range=[0, 1.15], gridcolor='#21262d', color='#8b949e'),
    xaxis=dict(color='#8b949e'),
    font=dict(family='monospace', size=12, color='#c9d1d9'),
    margin=dict(l=60, r=40, t=80, b=60),
    width=750, height=460, bargap=0.35,
)
fig3.show()


In [9]:
fig4 = go.Figure()
for c in CLUSTER_ORDER:
    fig4.add_trace(go.Violin(
        y=stats[c]['scores'],
        name=c,
        box_visible=True,
        meanline_visible=True,
        points='all',
        jitter=0.4,
        marker=dict(size=3, opacity=0.6, color=CLUSTER_COLOR2[c]),
        line_color=CLUSTER_COLOR2[c],
        fillcolor=CLUSTER_COLOR2[c],
        opacity=0.45,
    ))

fig4.add_hline(
    y=0.5, line_dash='dash', line_color='#f0e68c',
    annotation_text='Failure threshold  0.5',
    annotation_position='top right',
    annotation_font=dict(color='#f0e68c', size=11),
)
fig4.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    title=dict(text='Score Distribution per Cluster',
               font=dict(size=18, color='#58a6ff'), x=0.5),
    yaxis=dict(title='Quality Score', range=[-0.05, 1.1],
               gridcolor='#21262d', color='#8b949e'),
    xaxis=dict(color='#8b949e'),
    font=dict(family='monospace', size=12, color='#c9d1d9'),
    showlegend=False, violingap=0.3,
    margin=dict(l=60, r=40, t=80, b=60),
    width=750, height=460,
)
fig4.show()


---

## Step 6 — Run This on Real Data

Replace `all_vecs`, `all_scores`, and `all_prompts` above with real data from
your own evaluation pipeline:

```python
from faultmap import SliceAnalyzer

# Mode 1: bring your own pre-computed scores
analyzer = SliceAnalyzer(
    model='gpt-4o-mini',
    embedding_model='text-embedding-3-small',
)
report = analyzer.analyze(prompts, responses, scores=scores)
print(report)

# The report gives you failure slices with statistical p-values.
# Feed report.slices[i].sample_indices back into your embeddings
# to visualise exactly which points belong to each flagged cluster.
```

Full API docs, examples, and changelog:
[github.com/gabonavarroo/faultmap](https://github.com/gabonavarroo/faultmap)
